# Análise das métricas de prompt optimization do `ncf` para `etd`

Este notebook percorre os processos finalizados em `out/prompt_optimization/Llama3.1-I/ncf/etd` e ajuda a responder quais configurações tiveram melhor desempenho dentro da métrica de objetivo `etd`.

O foco aqui é:
- destacar a melhor combinação geral;
- mostrar a melhor combinação por representação, quando houver mais de uma;
- exibir o ranking completo das configurações;
- ligar os processos aos resultados de teste quando existir `responses_metadata.json` correspondente;
- plotar curvas agregadas de treino e validação para comparar os processos;
- e abrir uma seção detalhada por processo com curvas por época, prompts principais e tabela das épocas.

Observações:
- O melhor processo é definido pelo maior `best_train_metric`.
- Em caso de empate, o desempate usa `best_val_metric` e depois menor `time_prompt_optimization`.
- Este notebook fica restrito à métrica `etd`, então as comparações aqui já acontecem dentro do mesmo objetivo.
- O padrão de cores é fixo em todos os gráficos: `etd` em azul, `sep` em laranja e `sep_etd_f1` em verde.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

def _is_project_root(candidate: Path) -> bool:
    return (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists()


def find_local_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    descendant_hints = []
    for base in (start, *start.parents):
        descendant_hints.extend(
            [
                base / "prompt-optim-expl-rec" / "explainability-with-LLMs",
                base / "explainability-with-LLMs",
            ]
        )

    for candidate in descendant_hints:
        if _is_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Não foi possível localizar a raiz de explainability-with-LLMs. "
        "Execute o notebook no projeto, em um subdiretório dele ou a partir da raiz do workspace."
    )

PROJECT_ROOT = find_local_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.optimization_process_analysis import (
    discover_optimization_processes,
    load_process_bundle,
    summarize_processes,
)

METRIC_COLOR_MAP = {
    "etd": "#1f77b4",
    "sep": "#ff7f0e",
    "sep_etd_f1": "#2ca02c",
    "geom_mean": "#d62728",
    "mean_balance": "#8c564b",
}


def metric_color(metric_name: str | None, default: str = "#4C78A8") -> str:
    return METRIC_COLOR_MAP.get(str(metric_name), default)


METRIC_LABEL_MAP = {
    "etd": "ETD",
    "sep": "SEP",
    "sep_etd_f1": "SEP_ETD_F1",
    "geom_mean": "Geométrica",
    "mean_balance": "Mean Balance",
}


def metric_label(metric_name: str | None) -> str:
    return METRIC_LABEL_MAP.get(str(metric_name), str(metric_name))


def blend_with_white(color: str, blend: float) -> str:
    red, green, blue = mcolors.to_rgb(color)
    return mcolors.to_hex(
        tuple((1 - blend) * channel + blend for channel in (red, green, blue))
    )


def metric_shades(metric_name: str | None, size: int) -> list[str]:
    base_color = metric_color(metric_name)
    if size <= 1:
        return [base_color]

    max_blend = 0.45
    return [
        blend_with_white(base_color, max_blend * (index / max(1, size - 1)))
        for index in range(size)
    ]


def ensure_balance_metrics(epochs_df: pd.DataFrame) -> pd.DataFrame:
    epochs_df = epochs_df.copy()

    for split in ("train", "val"):
        sep_col = f"{split}_score_sep"
        etd_col = f"{split}_score_etd"

        if sep_col not in epochs_df.columns or etd_col not in epochs_df.columns:
            continue

        sep_series = pd.to_numeric(epochs_df[sep_col], errors="coerce")
        etd_series = pd.to_numeric(epochs_df[etd_col], errors="coerce")

        epochs_df[f"{split}_score_geom_mean"] = (
            sep_series.clip(lower=0) * etd_series.clip(lower=0)
        ).pow(0.5)
        epochs_df[f"{split}_score_mean_balance"] = (
            ((sep_series + etd_series) / 2.0)
            * (1 - (sep_series - etd_series).abs())
        )

    return epochs_df


def preferred_metric_order(metric_names: list[str]) -> list[str]:
    preferred = ["sep", "etd", "sep_etd_f1", "geom_mean", "mean_balance"]
    ordered = [metric_name for metric_name in preferred if metric_name in metric_names]
    ordered.extend(metric_name for metric_name in metric_names if metric_name not in ordered)
    return ordered

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 60)

PROJECT_ROOT


In [ ]:
ALGORITHM_NAME = "ncf"
OBJECTIVE_METRIC = "etd"
SEARCH_ROOT_REL = "out/prompt_optimization/Llama3.1-I/ncf/etd"
SEARCH_ROOT = PROJECT_ROOT / SEARCH_ROOT_REL
TEST_ROOT = PROJECT_ROOT / "out" / "test_explainability"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ALGORITHM_NAME:", ALGORITHM_NAME)
print("OBJECTIVE_METRIC:", OBJECTIVE_METRIC)
print("SEARCH_ROOT:", SEARCH_ROOT)
print("TEST_ROOT_EXISTS:", TEST_ROOT.exists())


In [ ]:
process_catalog = discover_optimization_processes(PROJECT_ROOT, search_root=SEARCH_ROOT)
if process_catalog.empty:
    raise FileNotFoundError("Nenhum optimization_process_metadata.json foi encontrado em " + str(SEARCH_ROOT))

process_catalog = process_catalog.copy()
process_catalog["early_label"] = "early_" + process_catalog["early_stopping"].astype(str).str.lower()
process_catalog["process_label"] = (
    process_catalog["representation_model"].astype(str)
    + " | "
    + process_catalog["early_label"].astype(str)
    + " | lambda="
    + process_catalog["mmr_lambda_quality"].astype(str)
    + " | pool="
    + process_catalog["mmr_pool_multiplier"].astype(str)
)
process_catalog["process_notebook_path_rel"] = process_catalog["process_dir_rel"] + "/plot_optimization_process.ipynb"

display(Markdown("## Catálogo dos processos encontrados em `" + SEARCH_ROOT_REL + "`"))
display(
    summarize_processes(
        process_catalog,
        columns=[
            "objective_metric",
            "representation_model",
            "early_stopping",
            "mmr_lambda_quality",
            "mmr_pool_multiplier",
            "epochs_completed",
            "best_train_epoch",
            "best_train_metric",
            "best_val_epoch",
            "best_val_metric",
            "saved_best_origin",
            "process_dir_rel",
        ],
    )
)
print("Quantidade de processos finalizados:", len(process_catalog))


In [ ]:
ranking_view = process_catalog.sort_values(
    by=["best_train_metric", "best_val_metric", "time_prompt_optimization"],
    ascending=[False, False, True],
    na_position="last",
).reset_index(drop=True)
ranking_view.insert(0, "rank", range(1, len(ranking_view) + 1))
best_row = ranking_view.iloc[0] if not ranking_view.empty else None

display(Markdown("## Melhor combinação geral"))
display(
    ranking_view.head(1).loc[
        :,
        [
            "rank",
            "process_label",
            "representation_model",
            "mmr_lambda_quality",
            "mmr_pool_multiplier",
            "best_train_epoch",
            "best_train_metric",
            "best_val_epoch",
            "best_val_metric",
            "time_prompt_optimization",
            "process_notebook_path_rel",
        ],
    ]
)
display(
    Markdown(
        "> Critério: `best_train_metric` decrescente. Desempates usam `best_val_metric` e depois menor `time_prompt_optimization`."
    )
)

display(Markdown("## Melhor combinação por representação"))
best_by_repr = (
    ranking_view.groupby("representation_model", as_index=False)
    .first()
    .loc[
        :,
        [
            "representation_model",
            "process_label",
            "mmr_lambda_quality",
            "mmr_pool_multiplier",
            "best_train_epoch",
            "best_train_metric",
            "best_val_epoch",
            "best_val_metric",
            "process_notebook_path_rel",
        ],
    ]
)
display(best_by_repr)

display(Markdown("## Ranking completo"))
ranking_columns = [
    "rank",
    "process_label",
    "representation_model",
    "early_stopping",
    "mmr_lambda_quality",
    "mmr_pool_multiplier",
    "best_train_epoch",
    "best_train_metric",
    "best_val_epoch",
    "best_val_metric",
    "epochs_completed",
    "time_prompt_optimization",
    "saved_best_origin",
    "process_notebook_path_rel",
]
display(ranking_view.loc[:, ranking_columns])

chart_df = ranking_view.dropna(subset=["best_train_metric"]).copy()
chart_colors = metric_shades(OBJECTIVE_METRIC, len(chart_df))
fig_height = max(4.0, 0.65 * len(chart_df))
fig, ax = plt.subplots(figsize=(13, fig_height))
ax.barh(chart_df["process_label"], chart_df["best_train_metric"], color=chart_colors)
ax.invert_yaxis()
ax.set_xlabel("best_train_metric")
ax.set_ylabel("processo")
ax.set_title("Melhores valores de treino")

max_value = chart_df["best_train_metric"].max()
offset = max_value * 0.01 if pd.notna(max_value) and max_value != 0 else 0.01
for index, value in enumerate(chart_df["best_train_metric"]):
    ax.text(value + offset, index, f"{value:.4f}", va="center")

plt.tight_layout()
plt.show()


In [ ]:
bundles = {}
linked_rows = []

for _, row in process_catalog.iterrows():
    bundle = load_process_bundle(Path(row["process_dir"]), PROJECT_ROOT)
    bundles[row["process_dir"]] = bundle
    linked_payload = bundle["linked_test_metadata"] or {}
    linked_rows.append(
        {
            "process_label": row["process_label"],
            "linked_test_metadata_path": bundle["summary"]["linked_test_metadata_path"],
            "test_metric_value": linked_payload.get("metric_value"),
            "n_users": linked_payload.get("n_users"),
            "time_to_explain": linked_payload.get("time_to_explain"),
            "prompt_source": linked_payload.get("prompt_source"),
        }
    )

linked_tests_df = pd.DataFrame(linked_rows)

display(Markdown("## Ligação com resultados de teste"))
if linked_tests_df["linked_test_metadata_path"].notna().any():
    display(linked_tests_df)
else:
    display(
        Markdown(
            "> Nenhum `responses_metadata.json` ligado aos `best_prompt.json` foi encontrado em `out/test_explainability`."
        )
    )


In [ ]:
if OBJECTIVE_METRIC == "sep_etd_f1":
    display(Markdown("## Curvas da melhor configuração global"))
    if best_row is None:
        display(Markdown("> Nenhum processo foi encontrado para a métrica `sep_etd_f1`."))
    else:
        display(
            best_row.loc[
                [
                    "rank",
                    "process_label",
                    "representation_model",
                    "mmr_lambda_quality",
                    "mmr_pool_multiplier",
                    "best_train_epoch",
                    "best_train_metric",
                    "best_val_epoch",
                    "best_val_metric",
                    "time_prompt_optimization",
                    "process_notebook_path_rel",
                ]
            ].to_frame(name="value")
        )

        epochs_df = ensure_balance_metrics(bundles[best_row["process_dir"]]["epochs_df"].copy())
        if epochs_df.empty:
            display(Markdown("> O melhor processo não possui histórico de épocas para plotar."))
        else:
            epoch_positions = epochs_df["epoch"] + 1
            combined_metric_names = preferred_metric_order(
                [
                    metric_name
                    for metric_name in ["sep", "etd", "sep_etd_f1", "geom_mean", "mean_balance"]
                    if (
                        f"train_score_{metric_name}" in epochs_df.columns
                        or f"val_score_{metric_name}" in epochs_df.columns
                    )
                ]
            )

            display(Markdown(
                "**Métricas derivadas neste notebook**: "
                "`geom_mean = sqrt(SEP * ETD)` e "
                "`mean_balance = ((SEP + ETD) / 2) * (1 - abs(SEP - ETD))`."
            ))

            fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharex=True, sharey=True)
            for ax, split_name, title in zip(
                axes,
                ["train", "val"],
                ["Treino", "Validação"],
            ):
                plotted_any = False
                for metric_name in combined_metric_names:
                    score_col = f"{split_name}_score_{metric_name}"
                    if score_col not in epochs_df.columns or not epochs_df[score_col].notna().any():
                        continue

                    ax.plot(
                        epoch_positions,
                        epochs_df[score_col],
                        marker="o",
                        linewidth=2,
                        color=metric_color(metric_name),
                        label=metric_label(metric_name),
                    )
                    plotted_any = True

                ax.set_title(f"{title}: comparação entre métricas")
                ax.set_xlabel("época")
                ax.set_ylabel("score")
                ax.set_xticks(epoch_positions.tolist())
                if plotted_any:
                    ax.legend(loc="best", ncol=2)
                else:
                    ax.text(0.5, 0.5, "Sem dados", ha="center", va="center", transform=ax.transAxes)

            plt.tight_layout()
            plt.show()

            metric_names = []
            for metric_name in [OBJECTIVE_METRIC, "sep", "etd"]:
                if metric_name not in metric_names:
                    metric_names.append(metric_name)

            for metric_name in metric_names:
                train_col = f"train_score_{metric_name}"
                val_col = f"val_score_{metric_name}"
                if train_col not in epochs_df.columns and val_col not in epochs_df.columns:
                    continue

                line_color = metric_color(metric_name)
                fig, ax = plt.subplots(figsize=(14, 6))

                if train_col in epochs_df.columns and epochs_df[train_col].notna().any():
                    ax.plot(
                        epoch_positions,
                        epochs_df[train_col],
                        marker="o",
                        linewidth=2,
                        linestyle="-",
                        color=line_color,
                        label=f"train::{metric_name}",
                    )

                if val_col in epochs_df.columns and epochs_df[val_col].notna().any():
                    ax.plot(
                        epoch_positions,
                        epochs_df[val_col],
                        marker="o",
                        linewidth=2,
                        linestyle="--",
                        color=line_color,
                        label=f"val::{metric_name}",
                    )

                ax.set_title(f"Evolução de {metric_name} na melhor configuração")
                ax.set_xlabel("época")
                ax.set_ylabel("score")
                ax.set_xticks(epoch_positions.tolist())
                ax.legend(loc="best")
                plt.tight_layout()
                plt.show()
else:
    display(Markdown("## Curvas agregadas por processo"))
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharex=True)
    train_ax, val_ax = axes
    plotted_train = False
    plotted_val = False

    curve_colors = metric_shades(OBJECTIVE_METRIC, len(ranking_view))

    for color, (_, row) in zip(curve_colors, ranking_view.iterrows()):
        epochs_df = bundles[row["process_dir"]]["epochs_df"]
        if epochs_df.empty:
            continue

        label = row["process_label"]
        if "train_metric" in epochs_df and epochs_df["train_metric"].notna().any():
            train_ax.plot(
                epochs_df["epoch"] + 1,
                epochs_df["train_metric"],
                marker="o",
                linewidth=2,
                color=color,
                label=label,
            )
            plotted_train = True

        if "val_metric" in epochs_df and epochs_df["val_metric"].notna().any():
            val_ax.plot(
                epochs_df["epoch"] + 1,
                epochs_df["val_metric"],
                marker="o",
                linewidth=2,
                color=color,
                label=label,
            )
            plotted_val = True

    train_ax.set_title("Treino")
    train_ax.set_xlabel("época")
    train_ax.set_ylabel("train_metric")

    val_ax.set_title("Validação")
    val_ax.set_xlabel("época")
    val_ax.set_ylabel("val_metric")

    if plotted_train:
        train_ax.legend(loc="best", fontsize=8)
    else:
        train_ax.text(0.5, 0.5, "Sem dados de treino", ha="center", va="center", transform=train_ax.transAxes)

    if plotted_val:
        val_ax.legend(loc="best", fontsize=8)
    else:
        val_ax.text(0.5, 0.5, "Sem dados de validação", ha="center", va="center", transform=val_ax.transAxes)

    plt.tight_layout()
    plt.show()


In [ ]:
if OBJECTIVE_METRIC == "sep_etd_f1":
    display(Markdown("## Detalhe da melhor configuração"))
    detail_iterator = ranking_view.head(1).iterrows()
else:
    display(Markdown("## Detalhe por processo"))
    detail_iterator = ranking_view.iterrows()

for _, row in detail_iterator:
    bundle = bundles[row["process_dir"]]
    summary = bundle["summary"]
    epochs_df = bundle["epochs_df"].copy()
    prompt_df = bundle["prompt_df"].copy()

    if OBJECTIVE_METRIC == "sep_etd_f1":
        epochs_df = ensure_balance_metrics(epochs_df)

    display(Markdown("### " + str(row["process_label"])))
    display(
        pd.DataFrame(
            [
                {
                    "process_dir_rel": summary["process_dir_rel"],
                    "representation_model": summary["representation_model"],
                    "early_stopping": summary["early_stopping"],
                    "mmr_lambda_quality": summary["mmr_lambda_quality"],
                    "mmr_pool_multiplier": summary["mmr_pool_multiplier"],
                    "best_train_epoch": summary["best_train_epoch"],
                    "best_train_metric": summary["best_train_metric"],
                    "best_val_epoch": summary["best_val_epoch"],
                    "best_val_metric": summary["best_val_metric"],
                    "saved_best_origin": summary["saved_best_origin"],
                    "linked_test_metadata_path": summary["linked_test_metadata_path"],
                }
            ]
        )
    )

    display(Markdown("**Prompts principais**"))
    display(prompt_df[["prompt_role", "epoch", "source_metric", "score", "prompt_preview"]])

    if not epochs_df.empty:
        epoch_positions = epochs_df["epoch"] + 1
        if OBJECTIVE_METRIC == "sep_etd_f1":
            metric_names = []
            for metric_name in [OBJECTIVE_METRIC, "sep", "etd"]:
                if metric_name not in metric_names:
                    metric_names.append(metric_name)

            for metric_name in metric_names:
                train_col = f"train_score_{metric_name}"
                val_col = f"val_score_{metric_name}"
                if train_col not in epochs_df.columns and val_col not in epochs_df.columns:
                    continue

                line_color = metric_color(metric_name)
                fig, ax = plt.subplots(figsize=(14, 6))
                if train_col in epochs_df.columns and epochs_df[train_col].notna().any():
                    ax.plot(
                        epoch_positions,
                        epochs_df[train_col],
                        marker="o",
                        linestyle="-",
                        color=line_color,
                        label=f"train:{metric_name}",
                    )

                if val_col in epochs_df.columns and epochs_df[val_col].notna().any():
                    ax.plot(
                        epoch_positions,
                        epochs_df[val_col],
                        marker="o",
                        linestyle="--",
                        color=line_color,
                        label=f"val:{metric_name}",
                    )

                ax.set_title(f"Evolução de {metric_name} por época")
                ax.set_xlabel("época")
                ax.set_ylabel("score")
                ax.set_xticks(epoch_positions.tolist())
                ax.legend(loc="best")
                plt.tight_layout()
                plt.show()
        else:
            metric_columns = sorted(
                {
                    column.replace("train_score_", "")
                    for column in epochs_df.columns
                    if column.startswith("train_score_")
                }
            )
            all_metric_names = []
            for metric_name in [summary["objective_metric"], *metric_columns]:
                if metric_name not in all_metric_names:
                    all_metric_names.append(metric_name)

            fig, ax = plt.subplots(figsize=(14, 6))
            for metric_name in all_metric_names:
                train_col = f"train_score_{metric_name}"
                val_col = f"val_score_{metric_name}"
                line_color = metric_color(metric_name)

                if train_col in epochs_df.columns and epochs_df[train_col].notna().any():
                    ax.plot(
                        epoch_positions,
                        epochs_df[train_col],
                        marker="o",
                        linestyle="-",
                        color=line_color,
                        label=f"train:{metric_name}",
                    )

                if val_col in epochs_df.columns and epochs_df[val_col].notna().any():
                    ax.plot(
                        epoch_positions,
                        epochs_df[val_col],
                        marker="o",
                        linestyle="--",
                        color=line_color,
                        label=f"val:{metric_name}",
                    )

            ax.set_title("Evolução das métricas por época")
            ax.set_xlabel("época")
            ax.set_ylabel("score")
            ax.set_xticks(epoch_positions.tolist())
            ax.legend(loc="best")
            plt.tight_layout()
            plt.show()

        epoch_columns = [
            "epoch",
            "generated_new_prompt",
            "train_metric",
            "val_metric",
            "val_improvement_vs_prev",
            "is_best_train_epoch",
            "is_best_val_epoch",
            "is_saved_best_epoch",
            "time_spent_instruction",
            "time_spent_train_eval",
            "time_spent_val_eval",
            "mmr_selected_reference_epochs",
            "prompt_preview",
        ]
        if OBJECTIVE_METRIC == "sep_etd_f1":
            score_columns = []
            for metric_name in [OBJECTIVE_METRIC, "sep", "etd", "geom_mean", "mean_balance"]:
                for column in (f"train_score_{metric_name}", f"val_score_{metric_name}"):
                    if column in epochs_df.columns and column not in score_columns:
                        score_columns.append(column)
        else:
            score_columns = sorted(
                [column for column in epochs_df.columns if column.startswith("train_score_") or column.startswith("val_score_")]
            )
        display(Markdown("**Tabela por época**"))
        display(epochs_df[epoch_columns + score_columns])
